In [1]:
!pip install -Uq langchain langchain-community langgraph yfinance langchain-openai langchain-text-splitters faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.6/259.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.1/146.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [8]:
from google.colab import userdata
import os
import yfinance as yf
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
import transformers
transformers.logging.set_verbosity_error()

from transformers import BertTokenizer, BertForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

import torch

# Configure the API key in environment variables
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_API_KEY')

## Ingest and Classify Tool

In [3]:
from langchain_core.tools import tool as ingest_tool

@ingest_tool
def ingest_and_analyze_stock(query: str):
    """Extracts the stock ticker from the query, fetches news, and analyzes sentiment."""
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
    llm = ChatOpenAI(model="gpt-4o-mini")

    # Use LLM to provide the stock ticker
    prompt = f"Extract the stock ticker symbol for the company mentioned in this query: '{query}'. Return ONLY the ticker symbol, nothing else."
    response = llm.invoke([HumanMessage(content=prompt)])
    ticker_symbol = response.content.strip()
    print(f"\nExtracted Ticker: {ticker_symbol}")

    # Fetch news for the ticker
    raw_news = yf.Search(ticker_symbol, max_results=5).news
    news_headlines = [item['title'] for item in raw_news if 'title' in item]

    print("\nNews Sentiment Analysis:")
    label_names = ["Negative", "Neutral", "Positive"]

    results = []
    # Use the transformer to classify the raw_news for sentiment analysis
    for headline in news_headlines:
        inputs = tokenizer(headline, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)

        predicted_labels = torch.argmax(outputs.logits, dim=1)
        predicted_label = label_names[predicted_labels.item()]

        result_str = f"- {headline} -> Sentiment: {predicted_label}"
        print(result_str)
        results.append(result_str)

    return "\n".join(results)


## Retrieval-Augmented Generation of Articles

In [4]:
import os
import torch
from transformers import pipeline
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
# from langchain.chains import RetrievalQA
from langchain_classic.chains import RetrievalQA

In [14]:
from langchain_core.tools import tool as rag_tool
from langchain_community.embeddings import HuggingFaceEmbeddings
import tempfile
import os

@rag_tool
def generate_rag_response(query: str) -> str:
    """Generates a response to a query using RAG."""
    llm = ChatOpenAI(model="gpt-4o-mini")

    # Use LLM to extract ticker symbol from query
    prompt = f"Extract the stock ticker symbol for the company mentioned in this query: '{query}'. Return ONLY the ticker symbol, nothing else."
    response = llm.invoke([HumanMessage(content=prompt)])
    ticker_symbol = response.content.strip()

    # Fetch news for the ticker
    raw_news = yf.Search(ticker_symbol, max_results=5).news

    # Convert news to string and save to temporary file for TextLoader
    news_content = "\n\n".join([f"Title: {item.get('title', '')}\nSummary: {item.get('summary', '')}" for item in raw_news if 'title' in item])

    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as temp_file:
        temp_file.write(news_content)
        temp_file_path = temp_file.name

    try:
        loader = TextLoader(temp_file_path, encoding="utf-8")
        documents = loader.load()
    finally:
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=200)
    docs = text_splitter.split_documents(documents[:8])

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db = FAISS.from_documents(docs, embeddings)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    qa_pipeline = pipeline(
        "text-generation",
        model="google/flan-t5-small",
        device=device,
        max_length=512,
        do_sample=False
    )

    llm_hf = HuggingFacePipeline(pipeline=qa_pipeline)

    retriever = db.as_retriever(search_kwargs={"k": 3})
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm_hf,
        retriever=retriever,
        return_source_documents=True
    )

    # Execute the QA chain for the final summary
    result = qa_chain({"query": query})
    summary = result.get("result", "")
    print(f"\nExtracted Article Summary: {summary}")

    return summary

## Re-routes to a new agent or tool when appropriate

In [6]:
from langchain_core.tools import tool as new_agent_tool

@new_agent_tool
def different_agent(query: str):
  """Takes questions not related to news and re-routes them to a different agent."""
  llm = ChatOpenAI(model="gpt-4o-mini")
  prompt = f"Recognize when another agent should take over when not news related from the query: '{query}'."
  other_agents = ""
  response = llm.invoke([HumanMessage(content=prompt)])
  return response.content


# Agent Tool calling

In [16]:
# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Provide the tool to the agent
tools = [ingest_and_analyze_stock, generate_rag_response, different_agent]

# Construct the Agent using LangGraph
agent_executor = create_react_agent(llm, tools)

# 3 questions designed to test each tool specifically
queries = [
    # 1. Tests `ingest_and_analyze_stock` (Sentiment Analysis)
    "What is the latest sentiment for Nvidia stock?",

    # 2. Tests `generate_rag_response` (RAG Summary)
    "Use the RAG tool to generate a summary response of the latest news for Apple stock.",

    # 3. Tests `different_agent` (Non-news routing) <---   (We need to adjust to pass off to other tools)
    "What are the current market trends?"
]

for query in queries:
    print(f"\n{'='*50}\nTesting Query: {query}\n{'='*50}")

    # Invoke the agent
    response = agent_executor.invoke({
        "messages": [HumanMessage(content=query)]
    })

    print("\n--- Agent Response ---")
    print(response["messages"][-1].content)



Testing Query: What are the current market trends?

--- Agent Response ---
I can't provide information on current market trends directly. For in-depth analysis, it would be best to consult a financial analyst or an economic expert who specializes in market conditions. If you have a specific market or sector in mind, I could help route the query accordingly.


## Checks what tool was called in the last query

In [17]:
# Checks what tool the llm called from query above.
for msg in response['messages']:
    print(f"\nMessage Type: {type(msg).__name__}")

    # If the AI decided to call a tool, it will be recorded in the 'tool_calls' attribute
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tool_call in msg.tool_calls:
            print(f"  -> Tool Selected: {tool_call['name']}")
            print(f"  -> Arguments Passed: {tool_call['args']}")

    # The actual output from the tool execution is stored in a ToolMessage
    elif type(msg).__name__ == 'ToolMessage':
        print(f"  -> Tool Output (truncated): {msg.content[:150]}...")


Message Type: HumanMessage

Message Type: AIMessage
  -> Tool Selected: different_agent
  -> Arguments Passed: {'query': 'current market trends'}

Message Type: ToolMessage
  -> Tool Output (truncated): In this case, the query "current market trends" is not news-related but instead pertains to an inquiry about ongoing market conditions or developments...

Message Type: AIMessage
